# 🏦 Loan Approval Prediction

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import pickle

In [ ]:
df = pd.read_csv("loan_data.csv")
df.head()

In [ ]:
# Handle missing values
num_cols = df.select_dtypes(include=np.number).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

cate_cols = df.select_dtypes(include='object').columns
df[cate_cols] = df[cate_cols].fillna(df[cate_cols].mode().iloc[0])

In [ ]:
# Encoding
encoders = {}
for col in cate_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le

In [ ]:
# Scaling
scaler = StandardScaler()
scale_cols = ['person_income', 'loan_amnt', 'credit_score', 'loan_percent_income']
df[scale_cols] = scaler.fit_transform(df[scale_cols])

In [ ]:
# Features
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Linear Regression
linear = LinearRegression()
linear.fit(X_train, y_train)

y_pred_linear = linear.predict(X_test)
y_pred_linear = [1 if i > 0.5 else 0 for i in y_pred_linear]

In [ ]:
# Decision Tree
dt = DecisionTreeClassifier(max_depth=5)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

In [ ]:
# Random Forest
rfc = RandomForestClassifier(n_estimators=100)
rfc.fit(X_train, y_train)

y_pred_rfc = rfc.predict(X_test)

In [ ]:
# Evaluation
print("Linear Regression Accuracy:", accuracy_score(y_test, y_pred_linear))
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rfc))

rmse = np.sqrt(mean_squared_error(y_test, y_pred_linear))
print("RMSE:", rmse)

In [ ]:
# Prediction
sample = X_test.iloc[0:1]
prediction = rfc.predict(sample)

print("Loan Approved" if prediction[0] == 1 else "Loan Rejected")

In [ ]:
# Save Model
export_data = {
    "model": rfc,
    "encoders": encoders,
    "scaler": scaler,
    "scale_cols": scale_cols,
    "cate_cols": cate_cols
}

with open("loan_model.pkl", "wb") as f:
    pickle.dump(export_data, f)

print("File saved successfully!")